
# Email Classification — End‑to‑End Data Analysis
**Author:** Business Analyst w/ Data Science expertise  
**Scope:** This notebook reproduces the full analysis and data manipulations we performed:
- Data loading & EDA
- Duplicate detection and cleaning (including conflicting labels)
- Class distribution (frequency breakdown)
- Balancing via RandomOverSampler
- Train/test split (75/25, stratified)
- Keyword/phrase extraction using TF‑IDF × χ² (with custom stop‑words)
- Overlap check across categories
- Cosine similarity between category centroids (to evaluate semantic proximity of `other`)
- Saving cleaned & balanced datasets and keyword dictionaries


In [ ]:

# --- Imports
import os
import re
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import chi2

# Try to import imblearn. If unavailable, we will skip oversampling gracefully.
try:
    from imblearn.over_sampling import RandomOverSampler
    IMBLEARN_OK = True
except Exception as e:
    print("[WARN] imblearn not available. Oversampling will be skipped. Reason:", e)
    IMBLEARN_OK = False

# Matplotlib defaults: single-plot charts, no color styles set
plt.rcParams.update({'figure.figsize': (8, 5), 'axes.grid': True})


In [ ]:

# --- Paths
from pathlib import Path
DATA_PATH = Path('data/synthetic_emails.csv')  # synthetic demo input; run from this project folder
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_PATH = OUTPUT_DIR / 'Dataset_cleaned.xlsx'
BALANCED_PATH = OUTPUT_DIR / 'Dataset_balanced.xlsx'
TRAIN_PATH = OUTPUT_DIR / 'Dataset_train_75.xlsx'
TEST_PATH  = OUTPUT_DIR / 'Dataset_test_25.xlsx'
KEYWORDS_HYBRID_TOP7_PATH = OUTPUT_DIR / 'Hybrid_Keywords_Top7.xlsx'

# --- Load dataset
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]
assert {'tagged_text','classification'}.issubset(df.columns), "Expected 'tagged_text' and 'classification' columns."

# Normalize base text
df['clean_text'] = df['tagged_text'].astype(str).str.strip().str.lower()

print("Loaded:", DATA_PATH)
df.head(3)

## 1. Basic EDA

In [ ]:

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)
print("\nMissing values per column:\n", df.isnull().sum())
print("\nSample rows:\n")
display(df.sample(min(5, len(df)), random_state=42))


## 2. Frequency breakdown by category

In [ ]:

freq = df['classification'].value_counts(dropna=False).reset_index()
freq.columns = ['Category', 'Count']
freq['Percentage'] = (freq['Count'] / freq['Count'].sum() * 100).round(1)
display(freq)

plt.figure()
order = freq.sort_values('Count', ascending=True)
plt.barh(order['Category'], order['Count'])
plt.title('Emails per category')
plt.xlabel('Count')
plt.ylabel('Category')
plt.tight_layout()
plt.show()


## 3. Duplicate detection and conflicting labels

In [ ]:

# Exact duplicates: same clean_text + same classification
dups_exact = df[df.duplicated(subset=['clean_text','classification'], keep=False)]
print(f"Exact duplicates found: {len(dups_exact)}")
display(dups_exact.sort_values(['classification','clean_text']).head(10))

# Conflicting labels: identical text under more than one label
conflict_texts = df.groupby('clean_text')['classification'].nunique()
conflict_keys = conflict_texts[conflict_texts > 1].index
conflicts_df = df[df['clean_text'].isin(conflict_keys)].sort_values('clean_text')
print(f"Conflicting texts (same content, multiple labels): {conflicts_df['clean_text'].nunique()}")
display(conflicts_df.head(10))


## 4. Cleaning — remove duplicates & conflicting label texts

In [ ]:

# Remove exact duplicates
df_nodup = df.drop_duplicates(subset=['clean_text','classification'], keep='first')

# Remove texts that appear under multiple labels
conflict_texts2 = df_nodup.groupby('clean_text')['classification'].nunique()
conflict_keys2 = conflict_texts2[conflict_texts2 > 1].index
df_cleaned = df_nodup[~df_nodup['clean_text'].isin(conflict_keys2)].copy()

print("Before cleaning:", df.shape, "After cleaning:", df_cleaned.shape)
df_cleaned.to_excel(CLEAN_PATH, index=False)
print("Saved cleaned dataset:", CLEAN_PATH)


## 5. Balancing via RandomOverSampler

In [ ]:

if IMBLEARN_OK:
    ros = RandomOverSampler(random_state=42)
    X = df_cleaned[['clean_text']]
    y = df_cleaned['classification']
    X_res, y_res = ros.fit_resample(X, y)
    df_balanced = pd.DataFrame({'clean_text': X_res['clean_text'], 'classification': y_res})
    df_balanced.to_excel(BALANCED_PATH, index=False)
    print("Saved balanced dataset:", BALANCED_PATH)
    display(df_balanced['classification'].value_counts())
else:
    print("[SKIP] Oversampling not performed (imblearn missing). Using cleaned dataset as fallback.")
    df_balanced = df_cleaned.copy()


## 6. Train/Test split (75/25, stratified)

In [ ]:

train_df, test_df = train_test_split(
    df_balanced,
    test_size=0.25,
    stratify=df_balanced['classification'],
    random_state=42
)
train_df.to_excel(TRAIN_PATH, index=False)
test_df.to_excel(TEST_PATH, index=False)

print("Train shape:", train_df.shape, "Test shape:", test_df.shape)


## 7. Keyword / Phrase extraction (TF‑IDF × χ², custom stop‑words)

In [ ]:

custom_stopwords = [
    'please','thank','thanks','hello','hi','regards','best',
    'you','your','my','we','us','me','account','number',
    'would','could','also','kindly','attached','looking',
    'i','the','to','in','of','for','on','a','an','and','is','was'
]

vectorizer = TfidfVectorizer(
    stop_words=custom_stopwords,
    ngram_range=(1,2),
    min_df=2,
    max_features=2000
)
X = vectorizer.fit_transform(df_balanced['clean_text'])
y = df_balanced['classification'].values
feature_names = np.array(vectorizer.get_feature_names_out())

le = LabelEncoder()
y_enc = le.fit_transform(y)

keywords_per_cat = {}
for i, cat in enumerate(le.classes_):
    mask = (y_enc == i)
    chi2_scores, _ = chi2(X, mask)
    tfidf_means = np.asarray(X[mask].mean(axis=0)).ravel()
    combined = chi2_scores * tfidf_means
    top_idx = np.argsort(combined)[::-1][:7]   # Top-7
    keywords_per_cat[cat] = feature_names[top_idx]

kw_df = pd.DataFrame(dict([(k, pd.Series(v)) for k,v in keywords_per_cat.items()]))
display(kw_df)
kw_df.to_excel(KEYWORDS_HYBRID_TOP7_PATH, index=False)
print("Saved:", KEYWORDS_HYBRID_TOP7_PATH)


## 8. Overlap check across categories

In [ ]:

from itertools import combinations

overlaps = {}
for (c1, w1), (c2, w2) in combinations(keywords_per_cat.items(), 2):
    inter = set(w1) & set(w2)
    if inter:
        overlaps[f"{c1} ↔ {c2}"] = sorted(list(inter))

if overlaps:
    print("Overlaps detected:")
    display(pd.DataFrame([(k, ', '.join(v)) for k,v in overlaps.items()], columns=['Pair','Overlap']))
else:
    print("No overlaps detected among top-7 keywords per category.")


## 9. Cosine similarity between category centroids

In [ ]:

cats = sorted(df_balanced['classification'].unique())
centroids = []
for c in cats:
    centroids.append(X[df_balanced['classification'].values == c].mean(axis=0))

import numpy as np
centroids = np.vstack(centroids)
sim_mat = cosine_similarity(centroids)
sim_df = pd.DataFrame(sim_mat, index=cats, columns=cats)
display(sim_df.round(3))

if 'other' in sim_df.index:
    closest = sim_df.loc('other') if callable(sim_df.loc) else sim_df.loc['other']
    closest = closest.drop('other').sort_values(ascending=False).head(3)
    print("Most similar to 'other':\n", closest)
else:
    print("Category 'other' not present.")

# Simple heatmap-like visualization
plt.figure(figsize=(6,5))
plt.imshow(sim_mat, aspect='auto')
plt.xticks(range(len(cats)), cats, rotation=45, ha='right')
plt.yticks(range(len(cats)), cats)
plt.title('Cosine similarity between category centroids')
plt.colorbar()
plt.tight_layout()
plt.show()


## 10. Artifacts & next steps

In [ ]:

artifacts = {
    'cleaned_dataset': CLEAN_PATH,
    'balanced_dataset': BALANCED_PATH,
    'train_split_75': TRAIN_PATH,
    'test_split_25': TEST_PATH,
    'keywords_hybrid_top7': KEYWORDS_HYBRID_TOP7_PATH
}
print(json.dumps(artifacts, indent=2))

print("\nNext steps:")
print("- Train baseline classifier (e.g., LogisticRegression/SVM) using TF-IDF (1,2)-grams.")
print("- Evaluate via stratified 5-fold CV; log per-class precision/recall.")
print("- Integrate language detection + translation if non-English emails expected.")
print("- Add validation_status column and filter invalid emails before training.")


In [ ]:

# --- Save all artifacts again (ensuring persistence)
from pathlib import Path
import shutil

artifacts = {
    'cleaned_dataset': CLEAN_PATH,
    'balanced_dataset': BALANCED_PATH,
    'train_split_75': TRAIN_PATH,
    'test_split_25': TEST_PATH,
    'keywords_hybrid_top7': KEYWORDS_HYBRID_TOP7_PATH
}

for name, path in artifacts.items():
    print(f"Artifact saved: {name} -> {path}")
